## Activation functions

#### Define a class with activation functions - sigmoid $\sigma(z)$, tanh(z), ReLU(z)

In [2]:
import pandas as pd
import numpy as np

#function to register new functions to a class
def add_to_class(Class): 
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

### Implement Sigmoid function

$\sigma (z) = \frac{1}{1 + e^{-z}}$ 

#### Computational considerations

- If z is a large positive number, then $e^{-z} = \frac{1}{e^z}$ is a small fraction that can safely become zero in floating point arithmetic. 

- If z is a large negative number, then -z is a large positive number, making $e^{-z}$ a large number that can cause numeric overflow. 

The sigmoid function has two mathematically equivalent forms. The implementation chooses between them based on the sign of the input to avoid numerical overflow when computing the exponential.

When z < 0, we use 

$\sigma (z) = \frac{e^{z}}{1 + e^{z}}$

else we use 

$\sigma (z) = \frac{1}{1 + e^{-z}}$ 

#### Handling vector arithmetic

Let's say z = np.array([-2, 0 3])

The check z >= 0 produces an array of boolean values: [False, True, True]

Since the sigmoid function needs to be applied to each member in the array, we can use the array of boolean values as a mask to decide which of the two forms of sigmoid function to use. Run the following example code to see how it will work.


In [4]:
z = np.array([-2, 0, 3])
positive_mask = z >= 0
negative_mask = ~positive_mask
print(positive_mask)
print(z[positive_mask])
print(z[negative_mask])

[False  True  True]
[0 3]
[-2]


In [5]:
class ActivationFunctions:
    def __init__(self):
        pass
    
    def act_sigmoid(self, z):
        output = np.empty_like(z, dtype=float)
        positive_mask = z >= 0
        negative_mask = ~positive_mask
        output[positive_mask] = 1 / ( 1 + np.exp(-z[positive_mask]))
        output[negative_mask] = np.exp(z[negative_mask]) / (1 + np.exp(z[negative_mask]))
        return output    


---

### Implement tanh

$tanh(z) = \frac{e^{z} - e^{-z}} {e^{z} + e^{-z}}$ 


To reuse the code for sigmoid above, we can write tanh in terms of sigmoid as:

$tanh(z) = 2 * \sigma (2z) -1 $

In [6]:
@add_to_class(ActivationFunctions)
def act_tanh(self, z):
    output = 2 * self.act_sigmoid(2*z) - 1
    return output

---

### Implement ReLU

$\text{ReLU}(z) = \max (0,z) = \begin{cases} z & z > 0 \\ 0 & z \leq 0\end{cases}$

In [7]:
@add_to_class(ActivationFunctions)
def act_ReLU(self, z):
    output = np.where(z>0, z, 0)
    return output

---

### Test activation functions

#### Apply the activation functions to the following input vector. The expected result for each function is shown below. </p>

$$\mathbf{v} = \left(10^{4}, 5, 0, -10, -10^{9}\right)$$

$$\sigma({\mathbf{v}}) = \left( 1, 0.993, 0.5, 0, 0 \right) $$

$$\tanh(\mathbf{v}) = \left(1, 1, 0, -1, -1\right)$$

$$\text{ReLU}(\mathbf{v}) = \left(10^{4}, 5, 0, 0, 0\right)$$

In [20]:
v = np.array([10**4, 5, 0, -10, -10**9])
act = ActivationFunctions()
print("Sigmoid v:", act.act_sigmoid(v))
print("Tanh v:", act.act_tanh(v))
print("ReLU v:", act.act_ReLU(v))

Sigmoid v: [1.00000000e+00 9.93307149e-01 5.00000000e-01 4.53978687e-05
 0.00000000e+00]
Tanh v: [ 1.         0.9999092  0.        -1.        -1.       ]
ReLU v: [10000     5     0     0     0]


---

### Write functions for derivatives of activation functions

$\sigma (z) = \frac{1}{1 + e^{-z}}$ 

$\frac{d\sigma}{dz} = \frac{e^{-z}}{(1+e^{-z})^2} = \sigma(z)^2 e^{-z} = \sigma(z)(1 - \sigma(z))$


In [28]:
@add_to_class(ActivationFunctions)
def dsig_dz(self, z):
    sigma = self.act_sigmoid(z)
    ddz = sigma * ( 1 - sigma)
    return ddz

$tanh(z) = \frac{e^{z} - e^{-z}} {e^{z} + e^{-z}}$

$\frac{d(tanh)}{dz} = 1 - (tanh(z))^2 $

In [29]:
@add_to_class(ActivationFunctions)
def dtanh_dz(self, z):
    tanh = self.act_tanh(z)
    ddz = 1 - tanh ** 2
    return ddz

$\text{ReLU}(x) = \max (0,x) = \begin{cases} x & x > 0 \\ 0 & x \leq 0\end{cases}$

In [30]:
@add_to_class(ActivationFunctions)
def dReLU_dz(self, z):
    ddz = np.where(z>0, 1, 0)
    return ddz

---

### Test derivative functions

In [33]:
act = ActivationFunctions()
print("Sigmoid derivative:", act.dsig_dz(v))
print("Tanh derivative:", act.dtanh_dz(v))
print("ReLU derivative:", act.dReLU_dz(v))

Sigmoid derivative: [0.00000000e+00 6.64805667e-03 2.50000000e-01 4.53958077e-05
 0.00000000e+00]
Tanh derivative: [0.00000000e+00 1.81583231e-04 1.00000000e+00 8.24461455e-09
 0.00000000e+00]
ReLU derivative: [1 1 0 0 0]
